In [ ]:
# Célula 1: Leitura dos PDFs dos discursos

import os
import re
from pathlib import Path
from typing import List, Dict, Any, Optional

import pandas as pd
from pypdf import PdfReader
from tqdm.auto import tqdm

# ==============================================================================
# 1. CONFIGURAÇÃO
# ==============================================================================

PASTA_LULA = "discursos_lula"
PASTA_BOLSONARO = "discursos_bolsonaro"

# ==============================================================================
# 2. FUNÇÕES AUXILIARES
# ==============================================================================

def extrair_texto_pdf(caminho_pdf: str) -> Optional[str]:
    """Extrai todo o texto de um arquivo PDF usando pypdf."""
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = []
        for pagina in reader.pages:
            texto_pagina = pagina.extract_text()
            if texto_pagina:
                texto_completo.append(texto_pagina)
        return "\n".join(texto_completo)
    except Exception as e:
        print(f"   ⚠️ Erro ao ler {caminho_pdf}: {e}")
        return None


def limpar_texto(texto: str) -> str:
    """Limpeza básica do texto extraído."""
    if not texto:
        return ""
    
    texto = re.sub(r'\n+', '\n', texto)      # Remove quebras de linha excessivas
    texto = re.sub(r' +', ' ', texto)        # Remove espaços múltiplos
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)  # Caracteres de controle
    return texto.strip()


def ler_pdfs_da_pasta(pasta: str, nome_conjunto: str) -> List[Dict[str, Any]]:
    """Percorre recursivamente uma pasta, lê todos os PDFs e retorna lista de dicionários."""
    dados = []
    caminho_pasta = Path(pasta)
    
    if not caminho_pasta.exists():
        print(f"❌ Pasta não encontrada: {pasta}")
        return dados
    
    arquivos_pdf = list(caminho_pasta.rglob("*.pdf"))
    print(f"📚 Lendo {len(arquivos_pdf)} PDFs de '{nome_conjunto}'...")
    
    for arquivo in tqdm(arquivos_pdf, desc=f"Processando {nome_conjunto}"):
        texto_bruto = extrair_texto_pdf(str(arquivo))
        if texto_bruto:
            texto_limpo = limpar_texto(texto_bruto)
            if texto_limpo:
                dados.append({
                    "arquivo": arquivo.name,
                    "caminho": str(arquivo),
                    "ano": arquivo.parent.name if arquivo.parent.name.isdigit() else None,
                    "texto": texto_limpo,
                })
    
    print(f"   ✅ {len(dados)} discursos de {nome_conjunto} processados.")
    return dados


# ==============================================================================
# 3. LEITURA
# ==============================================================================

discursos_lula = ler_pdfs_da_pasta(PASTA_LULA, "Lula")
discursos_bolsonaro = ler_pdfs_da_pasta(PASTA_BOLSONARO, "Bolsonaro")

# ==============================================================================
# 4. CRIAÇÃO DOS DATAFRAMES
# ==============================================================================

df_lula = pd.DataFrame(discursos_lula)
df_bolsonaro = pd.DataFrame(discursos_bolsonaro)

if not df_lula.empty:
    df_lula["presidente"] = "Lula"
if not df_bolsonaro.empty:
    df_bolsonaro["presidente"] = "Bolsonaro"

df_todos = pd.concat([df_lula, df_bolsonaro], ignore_index=True)

# ==============================================================================
# 5. RESUMO
# ==============================================================================

print("\n" + "="*60)
print("📊 RESUMO DAS BASES")
print("="*60)
print(f"Discursos do Lula:      {len(df_lula)}")
print(f"Discursos do Bolsonaro: {len(df_bolsonaro)}")
print(f"Total geral:            {len(df_todos)}")
print("="*60)

if not df_todos.empty:
    print("\n🔍 Amostra:")
    display(df_todos[["arquivo", "ano", "presidente", "texto"]].head(3))